# Laboratorio de Big Data y Analitica Avanzada — Unidad 2.2 (con Apache Spark)

Material de apoyo practico para la Unidad 2.2 (Big Data y Analitica Avanzada) del curso
Estrategias de Transformacion Digital.

Este notebook usa **Apache Spark real** (a traves de PySpark), no una simulacion.
Esta pensado para ejecutarse en Google Colab, que ya trae Java preinstalado, lo cual es
el unico requisito externo de Spark. Si se ejecuta en una maquina distinta a Colab, se
necesita tener Java 11 o superior instalado; todo lo demas (PySpark) se instala desde la
primera celda.

Contenido:

1. Instalacion y arranque de una sesion de Spark
2. Volumen: crear y explorar un DataFrame grande en Spark
3. Escalamiento: particiones, repartition y el plan de ejecucion
4. Analitica descriptiva: agregaciones con la API de DataFrame y con SQL
5. Analitica predictiva: un modelo de regresion con Spark MLlib
6. Analitica prescriptiva: priorizacion con funciones de ventana (window functions)
7. Arquitectura de datos por capas (bronze, silver, gold) con Parquet
8. Gobernanza y calidad de datos en Spark


## 1. Instalacion y arranque de una sesion de Spark

La primera celda instala PySpark. En Google Colab esto toma menos de un minuto; si
PySpark ya esta instalado (como seria el caso al ejecutar este notebook varias veces en
la misma maquina), la instalacion se salta automaticamente.

In [1]:
# Instala PySpark. El flag -q reduce el texto de salida, -U asegura la version mas reciente disponible
!pip install -q -U pyspark



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from pyspark.sql import SparkSession  # punto de entrada principal para trabajar con Spark
from pyspark.sql import functions as F  # funciones de Spark SQL (sum, avg, count, etc.), equivalentes a las de pandas
from pyspark.sql.window import Window  # permite definir funciones de ventana, usadas en la seccion de analitica prescriptiva
from pyspark.ml.feature import VectorAssembler  # combina varias columnas en un solo vector de features, requerido por Spark MLlib
from pyspark.ml.regression import LinearRegression  # modelo de regresion lineal de Spark MLlib
from pyspark.ml.evaluation import RegressionEvaluator  # calcula metricas de regresion sobre predicciones de Spark

import time  # para medir tiempos de ejecucion

# Crea (o recupera, si ya existe) una sesion de Spark.
# "local[*]" le dice a Spark que corra en esta misma maquina, usando todos los nucleos disponibles del procesador,
# en lugar de conectarse a un cluster remoto de varias maquinas
spark = (
    SparkSession.builder
    .appName("laboratorio_bigdata_u22")  # nombre identificador de esta aplicacion de Spark
    .master("local[*]")  # modo local, usando todos los nucleos disponibles
    .getOrCreate()  # crea la sesion si no existe, o reutiliza la ya creada
)

spark.sparkContext.setLogLevel("WARN")  # reduce el nivel de detalle de los mensajes internos de Spark en la consola

print("Version de Spark:", spark.version)  # confirma que la sesion quedo activa y que version de Spark se esta usando
print("Nucleos disponibles para Spark:", spark.sparkContext.defaultParallelism)  # numero de particiones que Spark usara por defecto


Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=36943 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=36943 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/02 22:27:03 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/10/02 22:27:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/claude/curso-transformacion-digital/venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/10/02 22:27:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version de Spark: 4.2.0
Nucleos disponibles para Spark: 2


## 2. Volumen: crear y explorar un DataFrame grande en Spark

Se genera un DataFrame de Spark de 5 millones de filas, simulando el mismo dataset de
transacciones de una cadena de tiendas usado en secciones anteriores del curso, pero
ahora construido y procesado directamente con la API de Spark en lugar de pandas.

In [3]:
n_transacciones = 5_000_000  # cinco millones de filas: un volumen que ya empieza a justificar el uso de Spark

# spark.range genera una columna "id" con valores de 0 a n_transacciones - 1, distribuida en particiones desde el inicio
transacciones_spark = spark.range(0, n_transacciones)

# Se agregan las demas columnas usando funciones de Spark SQL, aplicadas sobre toda la columna de una sola vez
# (F.rand() genera un numero aleatorio uniforme entre 0 y 1, distinto para cada fila)
transacciones_spark = (
    transacciones_spark
    .withColumn("tienda_idx", (F.rand(seed=42) * 4).cast("int"))  # indice aleatorio de tienda, entre 0 y 3
    .withColumn("categoria_idx", (F.rand(seed=43) * 4).cast("int"))  # indice aleatorio de categoria, entre 0 y 3
    .withColumn("monto", (F.rand(seed=44) * 200_000 + 10_000).cast("double"))  # monto aleatorio entre 10,000 y 210,000
    .withColumn("dia", (F.rand(seed=45) * 365).cast("int"))  # dia del anio, entre 0 y 364
)

# Se traducen los indices numericos a nombres legibles, usando una expresion condicional de Spark (equivalente a un CASE WHEN de SQL)
transacciones_spark = transacciones_spark.withColumn(
    "tienda",
    F.when(F.col("tienda_idx") == 0, "centro")
     .when(F.col("tienda_idx") == 1, "norte")
     .when(F.col("tienda_idx") == 2, "sur")
     .otherwise("oriente"),
).withColumn(
    "categoria",
    F.when(F.col("categoria_idx") == 0, "alimentos")
     .when(F.col("categoria_idx") == 1, "aseo")
     .when(F.col("categoria_idx") == 2, "tecnologia")
     .otherwise("ropa"),
).drop("tienda_idx", "categoria_idx")  # elimina las columnas de indice numerico, que ya no se necesitan

transacciones_spark.printSchema()  # muestra la estructura de columnas y tipos de dato del DataFrame de Spark
print("Numero de filas:", transacciones_spark.count())  # cuenta el total de filas (esto ya dispara una ejecucion distribuida real)


root
 |-- id: long (nullable = false)
 |-- monto: double (nullable = false)
 |-- dia: integer (nullable = true)
 |-- tienda: string (nullable = false)
 |-- categoria: string (nullable = false)



Numero de filas: 5000000


## 3. Escalamiento: particiones, repartition y el plan de ejecucion

A diferencia de pandas, en Spark los datos estan divididos en particiones desde el
principio, y las operaciones se ejecutan de forma distribuida sobre esas particiones.
Aqui se inspecciona cuantas particiones tiene el DataFrame, se cambia ese numero con
`repartition`, y se revisa el plan de ejecucion que Spark genera internamente antes de
ejecutar una agregacion — algo que pandas no tiene, porque pandas ejecuta cada
instruccion de inmediato, mientras que Spark primero construye un plan y lo optimiza
antes de ejecutarlo (evaluacion perezosa, o "lazy evaluation").

In [4]:
print("Particiones actuales:", transacciones_spark.rdd.getNumPartitions())  # numero de particiones con las que Spark dividio los datos

# Se fuerza un numero especifico de particiones, simulando como se repartirian los datos entre 8 nodos de un cluster
transacciones_repartido = transacciones_spark.repartition(8)
print("Particiones despues de repartition(8):", transacciones_repartido.rdd.getNumPartitions())  # confirma el nuevo numero de particiones


Particiones actuales: 2


Particiones despues de repartition(8): 8


In [5]:
consulta_agregacion = (
    transacciones_spark
    .groupBy("tienda", "categoria")  # agrupa por tienda y categoria, igual que se haria en pandas
    .agg(F.sum("monto").alias("total"))  # suma el monto por cada grupo
)

consulta_agregacion.explain(mode="formatted")  # muestra el plan de ejecucion que Spark construyo, SIN ejecutarlo todavia

# Solo al llamar a una accion (como .count(), .collect() o .show()) Spark ejecuta realmente el plan
inicio = time.time()  # marca de tiempo antes de ejecutar la accion
consulta_agregacion.show()  # esta es la accion que dispara la ejecucion distribuida real sobre las particiones
print(f"Tiempo de ejecucion de la agregacion: {time.time() - inicio:.2f} segundos")


== Physical Plan ==
AdaptiveSparkPlan (9)
+- HashAggregate (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Project (5)
            +- Project (4)
               +- Project (3)
                  +- Project (2)
                     +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 5000000, step=1, splits=Some(2))

(2) Project
Output [1]: [cast((rand(42) * 4.0) as int) AS tienda_idx#1]
Input [1]: [id#0L]

(3) Project
Output [2]: [tienda_idx#1, cast((rand(43) * 4.0) as int) AS categoria_idx#2]
Input [1]: [tienda_idx#1]

(4) Project
Output [3]: [tienda_idx#1, categoria_idx#2, ((rand(44) * 200000.0) + 10000.0) AS monto#3]
Input [2]: [tienda_idx#1, categoria_idx#2]

(5) Project
Output [3]: [monto#3, CASE WHEN (tienda_idx#1 = 0) THEN centro WHEN (tienda_idx#1 = 1) THEN norte WHEN (tienda_idx#1 = 2) THEN sur ELSE oriente END AS tienda#5, CASE WHEN (categoria_idx#2 = 0) THEN alimentos WHEN (categoria_idx#2 = 1) THEN aseo WHEN (categoria_idx#2 = 2) THEN tecnologia EL

+-------+----------+--------------------+
| tienda| categoria|               total|
+-------+----------+--------------------+
| centro|tecnologia|3.443263244894071E10|
|    sur| alimentos|3.439100168500819...|
| centro| alimentos|3.438961212466604...|
|    sur|tecnologia| 3.44169813303896E10|
| centro|      ropa|3.438581655760541E10|
|  norte| alimentos|3.444276834523353E10|
|    sur|      ropa|3.438414607695552E10|
|    sur|      aseo|3.434312456355206...|
| centro|      aseo|3.431165038068059...|
|oriente|tecnologia| 3.44600573044163E10|
|oriente| alimentos|3.435732024149039...|
|oriente|      aseo|3.430141681719864E10|
|  norte|      ropa|3.435533014621564...|
|oriente|      ropa|3.445647084477830...|
|  norte|tecnologia|3.427899406441319...|
|  norte|      aseo|3.429105226105959...|
+-------+----------+--------------------+

Tiempo de ejecucion de la agregacion: 1.61 segundos


El plan de ejecución (`explain`) es una de las piezas más importantes para trabajar en
serio con Spark: antes de mover o procesar un solo dato, Spark decide cómo va a
distribuir el trabajo entre las particiones, en qué orden va a aplicar cada operación, y
qué pasos puede optimizar o combinar — y solo ejecuta ese plan cuando se le pide
explícitamente un resultado (una acción), no cuando se define cada transformación. Esto
es justamente lo que permite a Spark decidir, de forma automática, la manera más
eficiente de distribuir una tarea grande entre varios nodos de un clúster.

## 4. Analitica descriptiva: agregaciones con la API de DataFrame y con SQL

Spark permite expresar la misma agregacion de dos formas equivalentes: con la API de
DataFrame (metodos encadenados, como en la seccion anterior) o con una consulta SQL
directa sobre una tabla temporal. Ambas formas generan el mismo plan de ejecucion
internamente.

In [6]:
resumen_api = (
    transacciones_spark
    .groupBy("tienda", "categoria")  # agrupa por tienda y categoria
    .agg(
        F.sum("monto").alias("total"),  # suma del monto por grupo
        F.avg("monto").alias("promedio"),  # promedio del monto por grupo
        F.count("*").alias("numero_transacciones"),  # conteo de transacciones por grupo
    )
    .orderBy(F.desc("total"))  # ordena de mayor a menor venta total
)

resumen_api.show(10)  # muestra las 10 combinaciones tienda-categoria con mayor venta total


+-------+----------+--------------------+------------------+--------------------+
| tienda| categoria|               total|          promedio|numero_transacciones|
+-------+----------+--------------------+------------------+--------------------+
|oriente|tecnologia| 3.44600573044163E10|110030.64410419462|              313186|
|oriente|      ropa|3.445647084477830...|110102.86323855178|              312948|
|  norte| alimentos|3.444276834523353E10|110004.59384047272|              313103|
| centro|tecnologia|3.443263244894071E10|110014.73710609783|              312982|
|    sur|tecnologia| 3.44169813303896E10|110028.71269306139|              312800|
|    sur| alimentos|3.439100168500819...|110072.69158142291|              312439|
| centro| alimentos|3.438961212466604...| 109948.2451712579|              312780|
| centro|      ropa|3.438581655760541E10| 109886.2226292987|              312922|
|    sur|      ropa|3.438414607695552E10| 110063.7834487472|              312402|
|oriente| alimen

In [7]:
transacciones_spark.createOrReplaceTempView("transacciones")  # registra el DataFrame como una tabla temporal, para poder consultarla con SQL

resumen_sql = spark.sql("""
    SELECT
        tienda,
        categoria,
        SUM(monto) AS total,
        AVG(monto) AS promedio,
        COUNT(*) AS numero_transacciones
    FROM transacciones
    GROUP BY tienda, categoria
    ORDER BY total DESC
""")  # la misma agregacion anterior, expresada como una consulta SQL estandar

resumen_sql.show(10)  # el resultado es idéntico al de la version con la API de DataFrame


+-------+----------+--------------------+------------------+--------------------+
| tienda| categoria|               total|          promedio|numero_transacciones|
+-------+----------+--------------------+------------------+--------------------+
|oriente|tecnologia| 3.44600573044163E10|110030.64410419462|              313186|
|oriente|      ropa|3.445647084477830...|110102.86323855178|              312948|
|  norte| alimentos|3.444276834523353E10|110004.59384047272|              313103|
| centro|tecnologia|3.443263244894071E10|110014.73710609783|              312982|
|    sur|tecnologia| 3.44169813303896E10|110028.71269306139|              312800|
|    sur| alimentos|3.439100168500819...|110072.69158142291|              312439|
| centro| alimentos|3.438961212466604...| 109948.2451712579|              312780|
| centro|      ropa|3.438581655760541E10| 109886.2226292987|              312922|
|    sur|      ropa|3.438414607695552E10| 110063.7834487472|              312402|
|oriente| alimen

## 5. Analitica predictiva: un modelo de regresion con Spark MLlib

Spark incluye su propia libreria de machine learning distribuido, MLlib, pensada para
entrenar modelos sobre datos que no caben en una sola maquina. La mecanica es similar a
scikit-learn (ensamblar features, entrenar, predecir, evaluar), pero las clases y el
flujo de trabajo son especificos de Spark. Aqui se entrena un modelo de regresion
lineal para predecir el monto de una transaccion a partir del dia del anio.

In [8]:
ensamblador = VectorAssembler(
    inputCols=["dia"],  # columnas de entrada que se combinaran en un solo vector de features
    outputCol="features",  # nombre de la columna de salida que contendra ese vector
)
datos_ml = ensamblador.transform(transacciones_spark).select("features", "monto")  # aplica el ensamblador y conserva solo lo necesario para el modelo

datos_entrenamiento, datos_prueba = datos_ml.randomSplit([0.7, 0.3], seed=42)  # divide los datos en 70% entrenamiento y 30% prueba

print("Filas de entrenamiento:", datos_entrenamiento.count())
print("Filas de prueba:", datos_prueba.count())


Filas de entrenamiento: 3500025


Filas de prueba: 1499975


In [9]:
regresion = LinearRegression(featuresCol="features", labelCol="monto")  # define el modelo, indicando cual columna son las features y cual es el objetivo
modelo_entrenado = regresion.fit(datos_entrenamiento)  # entrena el modelo distribuido sobre los datos de entrenamiento

predicciones = modelo_entrenado.transform(datos_prueba)  # genera predicciones sobre el conjunto de prueba
predicciones.select("features", "monto", "prediction").show(5)  # muestra algunas filas comparando el valor real contra la prediccion


26/10/02 22:27:51 WARN Instrumentation: [c08bac00] regParam is zero, which might cause numerical instability and overfitting.


+--------+------------------+------------------+
|features|             monto|        prediction|
+--------+------------------+------------------+
|   [0.0]|10150.184469731674|109930.45536216532|
|   [0.0]|10291.386081439992|109930.45536216532|
|   [0.0]|10312.069769679369|109930.45536216532|
|   [0.0]| 10334.91055636905|109930.45536216532|
|   [0.0]|10375.177942681914|109930.45536216532|
+--------+------------------+------------------+
only showing top 5 rows


In [10]:
evaluador = RegressionEvaluator(labelCol="monto", predictionCol="prediction", metricName="rmse")  # configura el evaluador para calcular RMSE
rmse = evaluador.evaluate(predicciones)  # calcula el RMSE sobre las predicciones del conjunto de prueba

evaluador_r2 = evaluador.setMetricName("r2")  # reconfigura el mismo evaluador para calcular R2 en lugar de RMSE
r2 = evaluador_r2.evaluate(predicciones)  # calcula el R2 sobre las mismas predicciones

print(f"RMSE: {rmse:,.0f}")
print(f"R2: {r2:.4f}")

print(
    "\nUn R2 cercano a 0 es el resultado esperado aqui: el monto se genero de forma "
    "totalmente aleatoria, sin relacion real con el dia del anio, asi que no hay "
    "ninguna tendencia real que el modelo pueda aprender. Esto en si mismo es un "
    "resultado valido e importante: confirma que, con estos datos, el dia del anio NO "
    "es un buen predictor del monto, en lugar de forzar una conclusion que los datos no "
    "respaldan."
)


RMSE: 57,731
R2: -0.0000

Un R2 cercano a 0 es el resultado esperado aqui: el monto se genero de forma totalmente aleatoria, sin relacion real con el dia del anio, asi que no hay ninguna tendencia real que el modelo pueda aprender. Esto en si mismo es un resultado valido e importante: confirma que, con estos datos, el dia del anio NO es un buen predictor del monto, en lugar de forzar una conclusion que los datos no respaldan.


## 6. Analitica prescriptiva: priorizacion con funciones de ventana

Se reproduce el ejercicio de asignar presupuesto de inventario por categoria, pero
ahora usando una funcion de ventana de Spark para calcular el ranking de cada categoria
segun su venta promedio, una tecnica estandar en SQL y Spark para este tipo de
problemas de priorizacion.

In [11]:
ventana_ranking = Window.orderBy(F.desc("promedio"))  # define una ventana que ordena todas las filas por promedio, de mayor a menor

venta_promedio_categoria = (
    transacciones_spark
    .groupBy("categoria")
    .agg(F.avg("monto").alias("promedio"))  # venta promedio por categoria
    .withColumn("ranking", F.rank().over(ventana_ranking))  # asigna un ranking a cada categoria segun su promedio, usando la ventana definida arriba
)

venta_promedio_categoria.orderBy("ranking").show()  # muestra las categorias ordenadas por su ranking de prioridad


26/10/02 22:28:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/02 22:28:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/02 22:28:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/02 22:28:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/02 22:28:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/02 22:28:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/02 22:28:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+------------------+-------+
| categoria|          promedio|ranking|
+----------+------------------+-------+
| alimentos|110016.15480900435|      1|
|tecnologia|110015.04447593211|      2|
|      ropa|110010.66959125412|      3|
|      aseo| 109956.7486939061|      4|
+----------+------------------+-------+



In [12]:
presupuesto_total = 50_000_000  # presupuesto disponible, igual que en el ejercicio anterior
costo_minimo_por_categoria = 5_000_000  # monto minimo garantizado por categoria

datos_ranking = venta_promedio_categoria.orderBy("ranking").collect()  # trae el resultado ya pequeno (4 filas) de Spark a Python, para aplicar la regla de asignacion

restante = presupuesto_total  # presupuesto que todavia no se ha asignado
asignaciones = []  # lista donde se guardara cuanto le toco a cada categoria

for fila in datos_ranking:  # recorre las categorias en orden de prioridad (ya ordenadas por Spark)
    categoria = fila["categoria"]  # nombre de la categoria en esta fila
    if restante < costo_minimo_por_categoria:  # si no queda presupuesto suficiente ni para el minimo
        asignaciones.append((categoria, 0))  # esta categoria no recibe presupuesto
        continue
    monto_asignado = min(costo_minimo_por_categoria * 2, restante)  # asigna hasta el doble del minimo, sin exceder lo restante
    asignaciones.append((categoria, monto_asignado))  # registra la asignacion
    restante -= monto_asignado  # descuenta del presupuesto restante

for categoria, monto in asignaciones:  # muestra el resultado final de la asignacion
    print(f"{categoria}: ${monto:,.0f}")
print(f"\nPresupuesto sin asignar: ${restante:,.0f}")


26/10/02 22:28:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/02 22:28:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


alimentos: $10,000,000
tecnologia: $10,000,000
ropa: $10,000,000
aseo: $10,000,000

Presupuesto sin asignar: $10,000,000


26/10/02 22:28:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/02 22:28:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/02 22:28:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/02 22:28:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 7. Arquitectura de datos por capas (bronze, silver, gold) con Parquet

Igual que en el ejercicio anterior, pero ahora las tres capas se escriben y se leen con
Spark directamente, que es como se haria en un data lakehouse real.

In [13]:
# --- Capa bronze: los datos crudos, con un par de problemas de calidad introducidos a proposito ---
datos_bronze = transacciones_spark  # parte del DataFrame ya generado
datos_bronze = datos_bronze.withColumn(
    "monto",
    F.when(F.col("id") == 0, F.lit(-500.0)).otherwise(F.col("monto")),  # simula un monto invalido en una sola fila
)
datos_bronze = datos_bronze.withColumn(
    "tienda",
    F.when(F.col("id") == 1, F.lit(None)).otherwise(F.col("tienda")),  # simula una tienda faltante en otra fila
)

datos_bronze.write.mode("overwrite").parquet("datos_lakehouse_spark/bronze")  # escribe la capa bronze en formato Parquet, sobrescribiendo si ya existe
print("Capa bronze escrita.")


Capa bronze escrita.


In [14]:
datos_bronze_leido = spark.read.parquet("datos_lakehouse_spark/bronze")  # lee de vuelta la capa bronze desde disco

datos_silver = (
    datos_bronze_leido
    .filter(F.col("monto") > 0)  # elimina transacciones con monto invalido
    .filter(F.col("tienda").isNotNull())  # elimina transacciones sin tienda registrada
)

filas_bronze = datos_bronze_leido.count()  # numero de filas antes de limpiar
filas_silver = datos_silver.count()  # numero de filas despues de limpiar
print(f"Filas eliminadas en la limpieza: {filas_bronze - filas_silver}")

datos_silver.write.mode("overwrite").parquet("datos_lakehouse_spark/silver")  # escribe la capa silver ya limpia
print("Capa silver escrita.")


Filas eliminadas en la limpieza: 2


Capa silver escrita.


In [15]:
datos_silver_leido = spark.read.parquet("datos_lakehouse_spark/silver")  # lee de vuelta la capa silver desde disco

datos_gold = (
    datos_silver_leido
    .groupBy("tienda", "categoria")
    .agg(
        F.sum("monto").alias("total"),
        F.avg("monto").alias("promedio"),
        F.count("*").alias("numero_transacciones"),
    )
)  # construye la tabla de negocio final a partir de la capa silver ya limpia

datos_gold.write.mode("overwrite").parquet("datos_lakehouse_spark/gold")  # escribe la capa gold, lista para un reporte o dashboard
print("Capa gold escrita.")

spark.read.parquet("datos_lakehouse_spark/gold").show()  # lee y muestra la capa gold final, para confirmar que quedo correcta


Capa gold escrita.


+-------+----------+--------------------+------------------+--------------------+
| tienda| categoria|               total|          promedio|numero_transacciones|
+-------+----------+--------------------+------------------+--------------------+
| centro|tecnologia|3.443263244894071E10|110014.73710609783|              312982|
|    sur| alimentos|3.439100168500819...|110072.69158142291|              312439|
| centro| alimentos|3.438961212466604...| 109948.2451712579|              312780|
|    sur|tecnologia|3.441679773804375...|110028.47751445418|              312799|
| centro|      ropa|3.438581655760541E10| 109886.2226292987|              312922|
|  norte| alimentos|3.444276834523353E10|110004.59384047272|              313103|
|    sur|      ropa|3.438397637026178E10| 110063.5925309515|              312401|
|    sur|      aseo|3.434312456355206...|109962.74466742678|              312316|
| centro|      aseo|3.431165038068059...|109887.65282386025|              312243|
|oriente|tecnolo

## 8. Gobernanza y calidad de datos en Spark

Las mismas validaciones de calidad de antes, ahora expresadas con operaciones de Spark,
que es como se ejecutarian sobre un volumen de datos que realmente requiere
procesamiento distribuido.

In [16]:
def validar_calidad_spark(df, nombre_capa):
    """Ejecuta validaciones basicas de calidad sobre un DataFrame de Spark."""
    print(f"--- Validacion de calidad: capa {nombre_capa} ---")

    total_filas = df.count()  # numero total de filas en esta capa

    for columna in df.columns:  # recorre cada columna del DataFrame
        nulos = df.filter(F.col(columna).isNull()).count()  # cuenta cuantas filas tienen valor nulo en esta columna
        if nulos > 0:  # solo reporta las columnas que realmente tienen nulos
            print(f"  Columna '{columna}': {nulos} valores nulos ({nulos / total_filas * 100:.4f} por ciento)")

    if "monto" in df.columns:  # esta validacion solo aplica si la tabla tiene columna de monto
        invalidos = df.filter(F.col("monto") <= 0).count()  # cuenta montos invalidos
        print(f"  Montos invalidos (<= 0): {invalidos}")

    print()  # linea en blanco para separar cada reporte


validar_calidad_spark(spark.read.parquet("datos_lakehouse_spark/bronze"), "bronze")  # valida la capa cruda
validar_calidad_spark(spark.read.parquet("datos_lakehouse_spark/silver"), "silver")  # valida la capa limpia
validar_calidad_spark(spark.read.parquet("datos_lakehouse_spark/gold"), "gold")  # valida la capa de negocio


--- Validacion de calidad: capa bronze ---


  Columna 'tienda': 1 valores nulos (0.0000 por ciento)


  Montos invalidos (<= 0): 1

--- Validacion de calidad: capa silver ---


  Montos invalidos (<= 0): 0

--- Validacion de calidad: capa gold ---


In [17]:
spark.stop()  # cierra la sesion de Spark y libera los recursos que estaba usando
print("Sesion de Spark cerrada.")


Sesion de Spark cerrada.


## Cierre

Este notebook recorrio los mismos conceptos de la Unidad 2.2, pero ahora con Apache
Spark real en lugar de una simulacion: creacion y exploracion de un DataFrame
distribuido, particiones y planes de ejecucion, los tres niveles de analitica
(descriptiva con la API de DataFrame y con SQL, predictiva con Spark MLlib, y
prescriptiva con funciones de ventana), arquitectura de datos por capas con Parquet, y
validaciones de calidad de datos sobre un DataFrame de Spark.

Para el proyecto final, el ejercicio equivalente es identificar si los datos reales de
la organizacion elegida ya justifican el uso de una herramienta como Spark, o si todavia
caben comodamente en las herramientas convencionales vistas en la Unidad 2.1.